In [1]:
import sys
sys.path.append('/work/lpdi/users/ymiao/code/newcode_0923/atomsurf/')
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf

with initialize(config_path="./conf/"):
    cfg = compose(config_name="config.yaml")
cfg.data_dir='/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/'
cfg.loader.num_workers = 20
cfg.loader.batch_size = 1
cfg.loader.pin_memory =True
cfg.cfg_graph.use_esm=True
OmegaConf.register_new_resolver("eval", eval)
OmegaConf.resolve(cfg)
import torch
torch.multiprocessing.set_sharing_strategy('file_system')
torch.set_num_threads(1)
import hydra
import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from atomsurf.tasks.pip.pl_model import PIPModule
from atomsurf.tasks.pip.data_loader import PIPDataModule,PIPDataset,SurfaceLoaderPIP,GraphLoaderPIP
import warnings
warnings.filterwarnings("ignore")

/tmp/ipykernel_2994324/1569196969.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="./conf/"):


In [2]:

datamodule = PIPDataModule(cfg)

# init model
model = PIPModule(cfg)
loadmodel=model.load_from_checkpoint('/work/lpdi/users/ymiao/code/sbatch/piplog/lightning_logs/version_18_pip_pronet_hmrencoder_testangle/checkpoints/epoch=12-accuracy_val=0.00.ckpt')
testloader=datamodule.test_dataloader()

In [5]:
batchdata

AtomBatch(num_graphs=0)

In [6]:
for batchdata in testloader:
    if batchdata.num_graphs>=1 :
        if '3fhi.pdb1.gz_1_A'in batchdata.id[0] and  '3fhi.pdb1.gz_1_B' in batchdata.id[0]:
            break

idx error ('3ean.pdb2.gz_1_C', '3ean.pdb2.gz_1_D', None, None)
idx error idx error('3ean.pdb1.gz_1_A', '3ean.pdb1.gz_1_B', None, None)
 ('3ean.pdb3.gz_1_E', '3ean.pdb3.gz_1_F', None, None)idx error
 ('3eao.pdb1.gz_1_A', '3eao.pdb1.gz_1_B', None, None)
idx error ('3eao.pdb2.gz_1_C', '3eao.pdb2.gz_1_D', None, None)
idx error ('3eao.pdb3.gz_1_E', '3eao.pdb3.gz_1_F', None, None)


In [56]:

# id
pos_idx_left=batchdata.idx_left[0][idx]

In [94]:
set(pos_idx_left.numpy())

{6, 7, 8, 9, 41, 43, 45, 46, 47, 48, 67, 68, 69, 70, 71, 72}

In [7]:
surface_1, graph_1 = loadmodel.model.encoder(graph=batchdata.graph_1, surface=batchdata.surface_1)
surface_2, graph_2 = loadmodel.model.encoder(graph=batchdata.graph_2, surface=batchdata.surface_2)

In [37]:
all_left_pred=[]
for i in range(0,len(graph_1.x)):
    in_batch_left =  graph_1.x[i].expand(len(graph_2.x), -1)
    in_batch = torch.cat((in_batch_left, graph_2.x), dim=1)
    out_one = loadmodel.model.top_net(in_batch)
    # out_one= out_one.max()
    out_one = torch.sigmoid(out_one).max()
    all_left_pred.append(out_one.detach().numpy())
all_right_pred=[]
for i in range(0,len(graph_2.x)):
    in_batch_left =  graph_2.x[i].expand(len(graph_1.x), -1)
    in_batch = torch.cat(( graph_1.x,in_batch_left), dim=1)
    out_one = loadmodel.model.top_net(in_batch)
    # out_one= out_one.max()
    out_one = torch.sigmoid(out_one).max()
    all_right_pred.append(out_one.detach().numpy())


In [33]:
all_left_pred=np.array(all_left_pred)
all_left_pred -=all_left_pred.min()
all_left_pred /= all_left_pred.max()
all_right_pred=np.array(all_right_pred)
all_right_pred -=all_right_pred.min()
all_right_pred /= all_right_pred.max()
# all_left_pred
# all_left_pred

In [9]:

pred_pos_idx_right=np.where(np.vstack(all_right_pred)>0.96)[0]
pred_pos_idx_right

NameError: name 'np' is not defined

In [ ]:
pred_pos_idx_left=np.where(np.vstack(all_left_pred)>0.97)[0]

In [10]:
import numpy as np
np.where(np.array(all_left_pred)==1)

(array([], dtype=int64),)

In [158]:
# set(pos_idx_left.numpy())
set(pos_idx_right.numpy())

{0,
 1,
 2,
 3,
 5,
 6,
 7,
 8,
 13,
 42,
 43,
 46,
 51,
 78,
 109,
 112,
 113,
 141,
 142,
 146,
 149}

In [17]:
idx=torch.where(batchdata.label[0]==1)[0]
all_results_left=[]
pos_idx_left=batchdata.idx_left[0][idx]
for i in pos_idx_left:
    in_batch_left =  graph_1.x[i].expand(len(graph_2.x), -1)
    in_batch = torch.cat((in_batch_left, graph_2.x), dim=1)
    out_one = loadmodel.model.top_net(in_batch)
    out_one = torch.sigmoid(out_one).max().round()
    all_results_left.append(out_one)
all_results_left = torch.vstack(all_results_left)
# all_results = torch.concatenate(all_results, dim=1)
# all_results_max = torch.max(all_results, dim=1).values
    # break

In [18]:
all_results_right=[]
pos_idx_right=batchdata.idx_right[0][idx]
for i in pos_idx_right:
    in_batch_right =  graph_2.x[i].expand(len(graph_1.x), -1)
    in_batch = torch.cat((in_batch_right, graph_1.x), dim=1)
    out_one = loadmodel.model.top_net(in_batch)
    out_one = torch.sigmoid(out_one).max().round()
    all_results_right.append(out_one)
all_results_right = torch.vstack(all_results_right)

In [19]:
pos_idx_right_pred=pos_idx_right[torch.where(all_results_right==1)[0]]
pos_idx_left_pred=pos_idx_left[torch.where(all_results_left==1)[0]]

In [104]:

import numpy as np
count=0
result=[]
for batchdata in testloader:
    if batchdata.num_graphs>=1 :
        if batchdata.g1_len[0]!= batchdata.g2_len[0] and abs(batchdata.g1_len[0]-batchdata.g2_len[0])>50:
            pred=loadmodel(batchdata)
            labels = torch.cat(batchdata.label).reshape(-1,1)
            labels = labels.detach().cpu().numpy()
            predictions = pred.detach().cpu().numpy()
            auroc = roc_auc_score(y_true=labels, y_score=predictions)
            proceessed_pred=torch.sigmoid(pred).round().detach().numpy()
            idx,_=np.where(labels==1)
            acc = (labels==proceessed_pred)[idx].sum()/len(idx)
            if auroc>0.9 and acc>0.8:
                print(acc, auroc,batchdata.g1_len,batchdata.g2_len,len(pred),batchdata.id)
                result.append([acc, auroc,batchdata,proceessed_pred])

0.82 0.9576 tensor([365]) tensor([196]) 100 [['2a06.pdb1.gz_1_C', '2a06.pdb1.gz_1_E']]
0.8518518518518519 0.9190672153635118 tensor([241]) tensor([99]) 54 [['2a06.pdb1.gz_1_D', '2a06.pdb1.gz_1_F']]
0.9259259259259259 0.9423868312757202 tensor([241]) tensor([99]) 54 [['2a06.pdb1.gz_1_Q', '2a06.pdb1.gz_1_S']]
0.9318181818181818 0.9973599632690542 tensor([234]) tensor([327]) 264 [['4b7x.pdb5.gz_1_I', '4b7x.pdb5.gz_1_J']]
0.8888888888888888 0.9244444444444444 tensor([379]) tensor([196]) 90 [['1bcc.pdb1.gz_1_C', '1bcc.pdb1.gz_1_E']]
0.8148148148148148 0.921124828532236 tensor([196]) tensor([59]) 108 [['1bcc.pdb1.gz_1_E', '1bcc.pdb1.gz_1_J']]
0.8407643312101911 0.9683151446306139 tensor([294]) tensor([236]) 314 [['4bcm.pdb2.gz_1_C', '4bcm.pdb2.gz_1_D']]
0.8301886792452831 0.9134923460306159 tensor([446]) tensor([33]) 106 [['1be3.pdb1.gz_1_A', '1be3.pdb1.gz_1_I']]
0.8125 0.9487847222222221 tensor([379]) tensor([196]) 96 [['1be3.pdb2.gz_1_C', '1be3.pdb2.gz_1_E']]
0.875 0.9340277777777777 tenso

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe6ce891af0>
Traceback (most recent call last):
  File "/work/lpdi/users/ymiao/atomsurf_h100/lib/python3.9/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/work/lpdi/users/ymiao/atomsurf_h100/lib/python3.9/site-packages/torch/utils/data/dataloader.py", line 1443, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/work/lpdi/users/ymiao/atomsurf_h100/lib/python3.9/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/work/lpdi/users/ymiao/atomsurf_h100/lib/python3.9/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
  File "/work/lpdi/users/ymiao/atomsurf_h100/lib/python3.9/multiprocessing/connection.py", line 931, in wait
    ready = selector.select(timeout)
  File "/work/lpdi/users/ymiao/atomsurf_h100/lib/python3.9/selectors.py", line 416, in se

In [96]:

for batchdata in testloader:
    if batchdata.num_graphs>=1 :
        if batchdata.g1_len[0]!= batchdata.g2_len[0] and abs(batchdata.g1_len[0]-batchdata.g2_len[0])>50 and '2a06.pdb1.gz_1_D'in batchdata.id[0] and  '2a06.pdb1.gz_1_F'in batchdata.id[0]:
            pred=loadmodel(batchdata)
            labels = torch.cat(batchdata.label).reshape(-1,1)
            labels = labels.detach().cpu().numpy()
            predictions = pred.detach().cpu().numpy()
            auroc = roc_auc_score(y_true=labels, y_score=predictions)
            break

In [112]:
import numpy as np
labels = torch.cat(batchdata.label).reshape(-1,1)
labels = labels.detach().cpu().numpy()
np.where(labels==1)

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        34, 35, 36, 37, 38, 39, 40, 41, 42, 43]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))

In [113]:
import numpy as np
idx,_=np.where(labels==1)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26])

In [101]:
set(batchdata.idx_right.numpy()[idx])

{40, 41, 43, 44, 45, 46, 48, 51, 52, 54, 55, 58, 59}

In [92]:
(labels==proceessed_pred)[idx].sum()/len(idx)

0.4830508474576271

In [87]:
proceessed_pred

(236, 1)

In [120]:
for r in result:
    batchdata= r[2]
    if '3fhi.pdb1.gz_1_A' in batchdata.id[0] and  '3fhi.pdb1.gz_1_B'in batchdata.id[0] :
        break
print(batchdata)

AtomBatch(id=[1], idx_left=[294], g1_len=[1], graph_1=DataRGraphBatch(
  num_res=[1],
  node_len=[1],
  features=Features(num_nodes=0, possible_nums={0}),
  edge_index=[2, 9796],
  edge_attr=[9796],
  node_pos=[336, 3],
  x=[336, 128],
  misc_features={ pronet_features=[1] },
  batch=[336],
  ptr=[2]
), label=[1], g2_len=[1], surface_1=DataSurfaceBatch(n_verts=[1], k_eig=[1], features=Features(num_nodes=0, possible_nums={0}), verts=[1593, 3], faces=[3198, 3], mass=[1593, 1593, nnz=1593], L=[1593, 1593, nnz=11187], evals=[128], evecs=[1593, 128], gradX=[1593, 1593, nnz=11187], gradY=[1593, 1593, nnz=11187], vnormals=[1593, 3], x=[1593, 128], batch=[1593], ptr=[2], bp_gs=<atomsurf.network_utils.communication.passing_utils.BPGraphBatch object at 0x7fe6f8ed1580>, bp_sg=<atomsurf.network_utils.communication.passing_utils.BPGraphBatch object at 0x7fe6f61d5670>), surface_2=DataSurfaceBatch(n_verts=[1], k_eig=[1], features=Features(num_nodes=0, possible_nums={0}), verts=[822, 3], faces=[1648, 

In [121]:
from Bio.PDB import PDBParser, PDBIO

labels = torch.cat(batchdata.label).reshape(-1,1)
labels = labels.detach().cpu().numpy()
idx,_=np.where(labels==1)
# Load the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)
            if count in batchdata.idx_left.numpy()[idx]:
                for atom in residue:
                    atom.set_bfactor(99.0)  #
            else:
                for atom in residue:
                    atom.set_bfactor(0.0)  #
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+"_gd.pdb")
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)
            if count in batchdata.idx_right.numpy()[idx]:
                for atom in residue:
                    atom.set_bfactor(99.0)  #
            else:
                for atom in residue:
                    atom.set_bfactor(0.0)  #
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+"_gd.pdb")

In [22]:
from Bio.PDB import PDBParser, PDBIO


# Load the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)
            if count in pos_idx_left_pred:
                for atom in residue:
                    atom.set_bfactor(99.0)  #
            else:
                for atom in residue:
                    atom.set_bfactor(0.0)  #
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+"_prednew.pdb")
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)
            if count in pos_idx_right_pred:
                for atom in residue:
                    atom.set_bfactor(99.0)  #
            else:
                for atom in residue:
                    atom.set_bfactor(0.0)  #
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+"_prednew.pdb")

In [38]:

from Bio.PDB import PDBParser, PDBIO


# Load the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_left_pred[count]*100)
                # if all_left_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+"_predall.pdb")


parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_right_pred[count]*100)
                # if all_right_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+"_predall.pdb")

In [155]:
all_right_pred

[array(0.9277865, dtype=float32),
 array(0.9570505, dtype=float32),
 array(0.9815685, dtype=float32),
 array(0.9924908, dtype=float32),
 array(0.990119, dtype=float32),
 array(0.9949142, dtype=float32),
 array(0.996012, dtype=float32),
 array(0.995252, dtype=float32),
 array(0.99442613, dtype=float32),
 array(0.98798496, dtype=float32),
 array(0.9785593, dtype=float32),
 array(0.9905855, dtype=float32),
 array(0.9778512, dtype=float32),
 array(0.9897887, dtype=float32),
 array(0.90811735, dtype=float32),
 array(0.38396943, dtype=float32),
 array(0.6957955, dtype=float32),
 array(0.26498488, dtype=float32),
 array(0.05543484, dtype=float32),
 array(0.05234911, dtype=float32),
 array(0.19550696, dtype=float32),
 array(0.15377049, dtype=float32),
 array(0.55308825, dtype=float32),
 array(0.6343599, dtype=float32),
 array(0.678502, dtype=float32),
 array(0.92001677, dtype=float32),
 array(0.81675696, dtype=float32),
 array(0.7780046, dtype=float32),
 array(0.8509464, dtype=float32),
 array

In [8]:
batchdata.idx_right

tensor([ 161,  138,  154,  ...,  844,  948, 1043])

In [51]:
from atom3d.datasets import LMDBDataset
test = LMDBDataset('/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/')

In [54]:
for testf in test:
    if '4b3h.pdb1.gz_1_D' in testf['atoms_neighbors'].subunit1[0]:
        break


In [63]:
set(testf['atoms_neighbors']['residue1'])

{131,
 132,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 228,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 239,
 240,
 241,
 242,
 243,
 244,
 245,
 246,
 2058,
 2060,
 2062,
 2064,
 2107,
 2108}